## 1. Import des bibliothèques et chargement des données

In [1]:
import import_ipynb
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from adultdata_preparation_evaluat import prepare_adult_dataset

x_train, x_test, y_train, y_test, mapping = prepare_adult_dataset()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             32561 non-null  float64
 1   workclass       32561 non-null  object 
 2   fnlwgt          32561 non-null  float64
 3   education       32561 non-null  object 
 4   education_num   32561 non-null  float64
 5   marital_status  32561 non-null  object 
 6   occupation      32561 non-null  object 
 7   relationship    32561 non-null  object 
 8   race            32561 non-null  object 
 9   sex             32561 non-null  object 
 10  capital_gain    32561 non-null  float64
 11  capital_loss    32561 non-null  float64
 12  hours_per_week  32561 non-null  float64
 13  native_country  32561 non-null  object 
 14  label           32561 non-null  object 
dtypes: float64(6), object(9)
memory usage: 3.7+ MB
Training set:
<=50K    19778
>50K      6270
Name: label, dtype: int64
Test

## 2. Définition de la fonction d'entrainement RIPPER

In [2]:
import wittgenstein as lw
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time

def train_ripper(x_train, y_train, x_test, y_test, prune_size=0.33, k=2, max_rules=50):
    
    # Préparer DataFrame
    train_data = x_train.copy()
    train_data['label'] = y_train.values

    test_data = x_test.copy()
    test_data['label'] = y_test.values

    # Création et entrainement du modèle
    ripper = lw.RIPPER(prune_size=prune_size, k=k, max_rules=max_rules)
    start_time = time.time()
    ripper.fit(train_data, class_feat='label', pos_class=1, neg_class=0)
    train_time = time.time() - start_time

    # Prédiction
    y_pred = ripper.predict(test_data)

    # Calcul métriques
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred, average='weighted'),
        "precision": precision_score(y_test, y_pred, average='weighted', zero_division=0),
        "recall": recall_score(y_test, y_pred, average='weighted', zero_division=0),
        "train_time": train_time
    }

    # Règles générées
    rules = ripper.ruleset_

    return ripper, y_pred, metrics, rules

## 3. Entrainement du modèle RIPPER

In [3]:
ripper_model, y_pred_ripper, ripper_metrics, ripper_rules = train_ripper(
    x_train, y_train, x_test, y_test
)

print("Résultats RIPPER")
print("Accuracy:", ripper_metrics["accuracy"])
print("F1-score:", ripper_metrics["f1"])
print("Precision:", ripper_metrics["precision"])
print("Recall:", ripper_metrics["recall"])

print("Règles générées :")
print(ripper_rules)


Résultats RIPPER
Accuracy: 0.8251189927836634
F1-score: 0.8137379635150377
Precision: 0.8139281957227054
Recall: 0.8251189927836634
Règles générées :
[[marital_status=2^education_num=11.0-13.0^occupation=3.0-4.0^hours_per_week=48.0-55.0] V [marital_status=2^education_num=>13.0^native_country=36.0-39.0^capital_gain=9999.9-19999.8] V [marital_status=2^education_num=>13.0^capital_loss=1742.4-2178.0] V [marital_status=2^education_num=11.0-13.0^occupation=3.0-4.0^hours_per_week=40.0-48.0^workclass=5] V [marital_status=2^education_num=11.0-13.0^occupation=3.0-4.0^workclass=4] V [marital_status=2^education_num=>13.0^occupation=3.0-4.0^workclass=4^native_country=36.0-39.0^relationship=0^hours_per_week=48.0-55.0] V [marital_status=2^occupation=8.0-10.0^education_num=>13.0] V [marital_status=2^education_num=11.0-13.0^capital_loss=1742.4-2178.0^workclass=4] V [marital_status=2^education_num=11.0-13.0^education=8.0-9.0^hours_per_week=40.0-48.0] V [marital_status=2^education_num=11.0-13.0^capital_g

## 4. Sauvegarde des résultats et des règles générées

In [4]:
import pandas as pd
import os
# Créer dossier adultdataset
os.makedirs("results/adultdataset", exist_ok=True)


# RIPPER
ripper_df = pd.DataFrame([ripper_metrics])
ripper_df["method"] = "RIPPER"

ripper_df.to_csv("results/adultdataset/ripper_metrics.csv", index=False)
# RIPPER RULES
with open("results/adultdataset/ripper_rules.txt", "w") as f:
    f.write(str(ripper_rules))

## 7. Fonction d'évaluation détaillée du modèle RIPPER

In [5]:
def evaluate_ripper(model, X_train, y_train, X_test, y_test, elapsed=None, ram=None):

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    f1_train = f1_score(y_train, y_train_pred, average="weighted")
    f1_test = f1_score(y_test, y_test_pred, average="weighted")

    exact_train = accuracy_score(y_train, y_train_pred)
    exact_test = accuracy_score(y_test, y_test_pred)

    complexity = len(model.ruleset_) if hasattr(model, "ruleset_") else 1

    return {
        "F1_train": f1_train,
        "F1_test": f1_test,
        "Train_Coverage": np.nan,
        "Train_Exactitude": exact_train,
        "Test_Coverage": np.nan,
        "Test_Exactitude": exact_test,
        "Conflit": np.nan,
        "Non_couverts": np.nan,
        "F1_rmaj": np.nan,
        "F1_rr": f1_test,
        "Complexité": complexity,
        "Temps": round(elapsed, 2) if elapsed is not None else None,
        "RAM_MB": round(ram, 2) if ram is not None else None
    }